In [1]:
import numpy as np
import jax
import jax.numpy as jnp
from scipy.integrate import odeint
from ibm_nb import indep_init, ibm_init
from fenrir import *
import fenrir2 as ff
from jax.config import config
import rodeo.ibm as ibm
from rodeo.kalmantv import predict, update

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
config.update("jax_enable_x64", True)

In [2]:
def fitz0(X_t, t, theta):
    a, b, c = theta
    V, R = X_t 
    return np.array([c*(V - V*V*V/3 + R), -1/c*(V - a + b*R)])

def ode_fun(X, t, theta):
    "FitzHugh-Nagumo ODE function for jax."
    a, b, c = theta
    p = len(X)//2
    V, R = X[0], X[p]
    return jnp.array([c*(V - V*V*V/3 + R),
                      -1/c*(V - a + b*R)])

def fitz(X_t, t, theta):
    "Fitz ODE written for jax"
    a, b, c = theta
    V, R = X_t[:, 0]
    return jnp.array([[c*(V - V*V*V/3 + R)],
                      [-1/c*(V - a + b*R)]])
def ode_funpad(X, t, theta):
    a, b, c = theta
    p = len(X)//2
    V, R = X[0], X[p]
    return jnp.array([V, c*(V - V*V*V/3 + R), 0,
                      R, -1/c*(V - a + b*R), 0])


In [3]:
# problem setup and intialization
n_deriv = 1  # Total state
n_meas = 2  # Total measures
n_deriv_prior = 3
n_obs = 40
n_dim_obs = 2

n_order = jnp.array([n_deriv_prior]*n_meas)

# it is assumed that the solution is sought on the interval [tmin, tmax].
tmin = 0.
tmax = 40.

# logprior parameters
theta_true = np.array([0.2, 0.2, 3]) # True theta
n_theta = len(theta_true)
phi_mean = np.zeros(n_theta)
phi_sd = np.log(10)*np.ones(n_theta) 

# The rest of the parameters can be tuned according to ODE
# For this problem, we will use
sigma = .1
sigma = jnp.array([sigma]*n_meas)

# Initial x0 for odeint
ode0 = np.array([-1., 1.])

# Initial x0 for jax
x0_block = jnp.array([[-1., 1., 0.], [1., 1/3, 0.]])
x0_state = x0_block.flatten()

# Initial W for jax non block
W = np.zeros((n_meas, jnp.sum(n_order)))
W[0, 1] = 1
W[1, n_deriv_prior+1] = 1
W = jnp.array(W)

W2 = jnp.array([[[0., 1., 0.]], [[0., 1., 0.]]])  # ODE LHS matrix
key = jax.random.PRNGKey(0)

# observations
tseq1 = np.linspace(tmin, tmax, n_obs+1)
exact = odeint(fitz0, ode0, tseq1, args=(theta_true,))
gamma = .2
e_t = np.random.default_rng(100).normal(loc=0.0, scale=1, size=exact.shape)
y_obs = exact + gamma*e_t
mean_obs = jnp.zeros((n_dim_obs,))
trans_obs = np.zeros((n_dim_obs, jnp.sum(n_order)))
trans_obs[0,0] = 1
trans_obs[1, n_deriv_prior] = 1
trans_obs = jnp.array(trans_obs)
var_obs = gamma**2*jnp.eye((n_dim_obs))

# jit double filter
df_jit = jax.jit(fenrir_filter, static_argnums=(0, 5))

In [4]:
trans_obs2 = jnp.array([[[1., 0., 0.]], [[1., 0., 0.]]])
mean_obs2 = jnp.zeros((2, 1))
var_obs2 = gamma**2*jnp.array([[1.],[1.]])

In [5]:
trans_obs

DeviceArray([[1., 0., 0., 0., 0., 0.],
             [0., 0., 0., 1., 0., 0.]], dtype=float64)

In [6]:
# Get parameters for non block
n_res = 10
n_eval = n_obs*n_res
dt = (tmax-tmin)/n_eval
ode_init = ibm_init(dt, n_order, sigma)
kinit = indep_init(ode_init, n_order)
ode_init = dict((k, jnp.array(v)) for k, v in kinit.items())

ode_init2 = ibm.ibm_init(dt, n_order, sigma)

In [7]:
ff.fenrir_filter(fitz, W2, x0_block, theta_true, tmin, tmax, n_res,
              ode_init2['trans_state'], ode_init2['mean_state'], ode_init2['var_state'],
              trans_obs2, mean_obs2, var_obs2, y_obs)

DeviceArray(-12.1372778, dtype=float64)

In [8]:
fenrir_filter(ode_fun, W, x0_state, theta_true, tmin, tmax, n_res,
              ode_init['trans_state'], ode_init['mean_state'], ode_init['var_state'],
              trans_obs, mean_obs, var_obs, y_obs)

DeviceArray(-12.1372778, dtype=float64)

In [ ]:
res_lst = np.array([15,20]) 
n_samples = 100000
# phi_init = jnp.log(theta_true)
phi_init = np.append(np.log(theta_true), ode0)
theta_filter = np.zeros((len(res_lst), n_samples, n_theta+2))
for r,n_res in enumerate(res_lst):
    n_eval = n_obs*n_res
    y_out = jnp.ones((n_eval+1, n_dim_obs))*jnp.nan
    for i in range(n_obs+1):
        y_out = y_out.at[i*n_res].set(y_obs[i])

    # Get parameters for non block
    dt = (tmax-tmin)/n_eval
    ode_init = ibm_init(dt, n_order, sigma)
    kinit = indep_init(ode_init, n_order)
    ode_init = dict((k, jnp.array(v)) for k, v in kinit.items())
    
    inf = filter_inference(key, tmin, tmax, ode_fun)
    inf.W = W
    inf.mu_state = ode_init['mu_state']
    inf.wgt_state = ode_init['wgt_state']
    inf.var_state = ode_init['var_state']
    inf.mu_obs = mu_obs
    inf.wgt_obs = wgt_obs
    inf.var_obs = var_obs
    inf.funpad = ode_funpad

    phi_hat, phi_var, phi_fisher = inf.phi_fit(y_out, np.array([None, None]), phi_mean, phi_sd, phi_init, inf.double_filter_nlpost,
                                               double=True, varzero=True)
    
    theta_filter[r] = inf.phi_sample(phi_hat, phi_var, n_samples)
    theta_filter[r, :, :n_theta] = np.exp(theta_filter[r, :, :n_theta])
# np.save('saves/fitz_theta_filter.npy', theta_filter)